# 🔧 Advanced Feature Engineering Pipeline
## Cervical Cancer Risk Prediction - Train/Validation/Test Split

**Author:** ML Pipeline  
**Date:** 2026-07-10  
**Purpose:** Prepare data with 70%/15%/15% split for proper model validation

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import joblib

# Paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "kag_risk_factors_cervical_cancer.csv"
OUTPUT_DIR = PROJECT_ROOT / "processed_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Libraries imported successfully")
print(f"✓ Output directory: {OUTPUT_DIR}")

## 2. Load and Clean Data

In [ ]:
print("=" * 70)
print("STEP 1: LOAD AND CLEAN DATA")
print("=" * 70)

# Load dataset
df = pd.read_csv(DATA_PATH)
print(f"✓ Loaded dataset: {df.shape[0]} rows × {df.shape[1]} columns")

# Clean column names and convert the raw values to numeric.
df.columns = df.columns.str.strip()
df = df.replace('?', np.nan)
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

missing_counts = df.isna().sum()
print(f"✓ Data cleaned: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"✓ Missing values retained for fold-local imputation: {missing_counts[missing_counts > 0].to_dict()}")
df.head()

## 3. Feature Selection

In [ ]:
print("=" * 70)
print("STEP 2: FEATURE SELECTION")
print("=" * 70)

# Define target variable
target_col = 'Biopsy'

if target_col not in df.columns:
    raise ValueError(f"Target column '{target_col}' not found in dataset")

# Separate features and target
X = df.drop(columns=[target_col])
y = df[target_col]

print(f"✓ Features selected: {X.shape[1]} features")
print(f"✓ Target variable: {target_col}")
print(f"  - Class 0 (Negative): {(y == 0).sum()} samples")
print(f"  - Class 1 (Positive): {(y == 1).sum()} samples")
print(f"  - Imbalance ratio: {(y == 0).sum() / (y == 1).sum():.2f}:1")

## 4. Train/Validation/Test Split (70%/15%/15%)

**Improved split ratio:** More training data (70% vs 64%) for better model learning,
while keeping enough validation and test data for reliable evaluation.
Combined with 5-fold cross-validation in GridSearchCV for robust hyperparameter tuning.

In [ ]:
print("=" * 70)
print("STEP 3: TRAIN/VALIDATION/TEST SPLIT (70%/15%/15%)")
print("=" * 70)

# First split: 70% train, 30% temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Split the temporary set evenly into validation and test partitions.
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"✓ Train set:      {X_train.shape[0]:4d} samples ({X_train.shape[0]/len(X)*100:5.1f}%)")
print(f"✓ Validation set: {X_val.shape[0]:4d} samples ({X_val.shape[0]/len(X)*100:5.1f}%)")
print(f"✓ Test set:       {X_test.shape[0]:4d} samples ({X_test.shape[0]/len(X)*100:5.1f}%)")
print(f"✓ Total:          {len(X):4d} samples")

## 5. Feature Scaling

In [ ]:
print("=" * 70)
print("STEP 4: DEFER LEARNED PREPROCESSING")
print("=" * 70)

print("✓ Imputation and scaling are intentionally deferred to model pipelines.")
print("✓ Each CV fold learns preprocessing only from its training partition.")

## 6. Handle Class Imbalance (SMOTE)

In [ ]:
print("=" * 70)
print("STEP 5: RESAMPLING DEFERRED TO CROSS-VALIDATION")
print("=" * 70)

print("SMOTE is intentionally not applied during feature engineering.")
print("The training pipeline fits SMOTE independently inside each CV training fold.")
print(f"  - Class 0: {(y_train == 0).sum()} samples")
print(f"  - Class 1: {(y_train == 1).sum()} samples")
print("✓ Raw split retained to prevent synthetic samples crossing CV folds")

## 7. Save Processed Data

In [ ]:
print("=" * 70)
print("STEP 6: SAVE RAW SPLIT DATA")
print("=" * 70)

# Save unscaled, unresampled arrays. The model pipeline owns imputation,
# scaling, and resampling so every CV fold learns them independently.
np.save(OUTPUT_DIR / 'X_train.npy', X_train.to_numpy(dtype=np.float64))
np.save(OUTPUT_DIR / 'X_val.npy', X_val.to_numpy(dtype=np.float64))
np.save(OUTPUT_DIR / 'X_test.npy', X_test.to_numpy(dtype=np.float64))
np.save(OUTPUT_DIR / 'y_train.npy', y_train.to_numpy(dtype=np.int64))
np.save(OUTPUT_DIR / 'y_val.npy', y_val.to_numpy(dtype=np.int64))
np.save(OUTPUT_DIR / 'y_test.npy', y_test.to_numpy(dtype=np.int64))

with open(OUTPUT_DIR / 'feature_columns.json', 'w') as f:
    json.dump(list(X.columns), f, indent=2)

print(f"✓ Saved raw split data:")
print(f"  - X_train.npy: {X_train.shape}")
print(f"  - X_val.npy:   {X_val.shape}")
print(f"  - X_test.npy:  {X_test.shape}")
print(f"  - feature_columns.json: {len(X.columns)} features")

## 8. Summary

In [ ]:
print("=" * 70)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 70)

total = X_train.shape[0] + X_val.shape[0] + X_test.shape[0]

print(f"\n📊 Final Dataset Split:")
print(f"   - Training:      {X_train.shape[0]:4d} samples ({X_train.shape[0]/total*100:5.1f}%)")
print(f"   - Validation:    {X_val.shape[0]:4d} samples ({X_val.shape[0]/total*100:5.1f}%)")
print(f"   - Test:          {X_test.shape[0]:4d} samples ({X_test.shape[0]/total*100:5.1f}%)")

print(f"\n🎯 Target Distribution:")
print(f"   - Training:   {(y_train == 0).sum()} negative, {(y_train == 1).sum()} positive")
print(f"   - Validation: {(y_val == 0).sum()} negative, {(y_val == 1).sum()} positive")
print(f"   - Test:       {(y_test == 0).sum()} negative, {(y_test == 1).sum()} positive")

print("\n✅ Preprocessing contract:")
print("   1. Raw values cleaned and missing values retained")
print("   2. Stratified train/validation/test split")
print("   3. Imputation, scaling, and SMOTE occur inside training CV folds")
print("   4. Validation and test sets remain untouched")
print("\nNext step: Run 03_model_training_advanced.ipynb")